# 01 — External Data Extraction (100% on Kaggle)
**Purpose:** Extract ALL external features directly on Kaggle.

- **Part A:** API calls (Elevation, SoilGrids, Weather, OSM)
- **Part B:** Heavy file download -> extract per station -> delete

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00

**Output:** `train_enriched.parquet`, `val_enriched.parquet`

### Figures
1. Features per data source (bar chart)
2. Data coverage map per source (green/red per station)
3. Extraction summary table

> **Enable Internet** in Kaggle settings before running!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, time, requests
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11,
                     'axes.titleweight': 'bold', 'figure.dpi': 120})

SEED = 42
WORK_DIR  = '/kaggle/working'
CKPT_DIR  = f'{WORK_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Column config (must match notebook 00)
LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
DATE_COL    = 'Sample Date'
STATION_COL = 'station_id'

In [ ]:
# Load outputs from notebook 00
train_base = pd.read_parquet(f'{WORK_DIR}/train_base.parquet')
val_base   = pd.read_parquet(f'{WORK_DIR}/val_base.parquet')

print(f'Train: {train_base.shape},  Val: {val_base.shape}')
print(f'Train columns: {train_base.columns.tolist()}')

# Combine for per-station extraction
all_data = pd.concat([train_base, val_base], ignore_index=True)
unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()
print(f'\nUnique stations to extract: {len(unique_stations)}')

---
## Part A: API Extraction

### A1. Elevation (OpenTopoData)

In [ ]:
def fetch_elevation(lat, lon, retries=3):
    for attempt in range(retries):
        try:
            r = requests.get('https://api.opentopodata.org/v1/aster30m',
                           params={'locations': f'{lat},{lon}'}, timeout=30)
            r.raise_for_status()
            return float(r.json()['results'][0]['elevation'])
        except Exception:
            time.sleep(2 ** attempt)
    return np.nan

ckpt = f'{CKPT_DIR}/elevation.parquet'
if os.path.exists(ckpt):
    elev_df = pd.read_parquet(ckpt)
    print(f'[LOADED] elevation: {len(elev_df)} rows')
else:
    print(f'Extracting elevation for {len(unique_stations)} stations...')
    elev_df = unique_stations.copy()
    elev_df['elevation_m'] = elev_df.apply(
        lambda r: fetch_elevation(r[LAT_COL], r[LON_COL]), axis=1)
    elev_df.to_parquet(ckpt, index=False)
    print(f'[SAVED] nulls = {elev_df["elevation_m"].isnull().sum()}')

display(elev_df.describe())

### A2. SoilGrids (ISRIC)

In [ ]:
def fetch_soilgrids(lat, lon):
    props = ['phh2o', 'clay', 'sand', 'silt', 'ocd', 'cec']
    out = {}
    for prop in props:
        try:
            r = requests.get('https://rest.isric.org/soilgrids/v2.0/properties/query',
                           params={'lat': lat, 'lon': lon, 'property': prop,
                                   'depth': '0-5cm', 'value': 'mean'}, timeout=30)
            r.raise_for_status()
            val = r.json()['properties']['layers'][0]['depths'][0]['values']['mean']
            out[f'soil_{prop}'] = float(val) if val else np.nan
        except Exception:
            out[f'soil_{prop}'] = np.nan
    return out

ckpt = f'{CKPT_DIR}/soilgrids.parquet'
if os.path.exists(ckpt):
    soil_df = pd.read_parquet(ckpt)
    print(f'[LOADED] soilgrids: {len(soil_df)} rows')
else:
    print(f'Extracting SoilGrids for {len(unique_stations)} stations...')
    rows = unique_stations.apply(lambda r: fetch_soilgrids(r[LAT_COL], r[LON_COL]), axis=1)
    soil_df = pd.concat([unique_stations.reset_index(drop=True),
                         pd.DataFrame(rows.tolist())], axis=1)
    soil_df.to_parquet(ckpt, index=False)
    print(f'[SAVED] {len(soil_df)} rows')

display(soil_df.describe())

### A3. Weather (Open-Meteo archive, 7/14/30-day lag)

In [ ]:
def fetch_weather(lat, lon, date_str, lag=7):
    end = pd.to_datetime(date_str)
    start = end - timedelta(days=lag)
    try:
        r = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
            'latitude': lat, 'longitude': lon,
            'start_date': start.strftime('%Y-%m-%d'),
            'end_date': end.strftime('%Y-%m-%d'),
            'daily': 'precipitation_sum,temperature_2m_max,temperature_2m_min,windspeed_10m_max',
            'timezone': 'Africa/Johannesburg'
        }, timeout=30)
        r.raise_for_status()
        d = r.json().get('daily', {})
        precip = [p for p in d.get('precipitation_sum', []) if p is not None]
        tmax = [t for t in d.get('temperature_2m_max', []) if t is not None]
        tmin = [t for t in d.get('temperature_2m_min', []) if t is not None]
        wind = [w for w in d.get('windspeed_10m_max', []) if w is not None]
        return {
            f'precip_sum_{lag}d': sum(precip) if precip else np.nan,
            f'precip_max_{lag}d': max(precip) if precip else np.nan,
            f'temp_max_{lag}d': max(tmax) if tmax else np.nan,
            f'temp_min_{lag}d': min(tmin) if tmin else np.nan,
            f'temp_range_{lag}d': (max(tmax)-min(tmin)) if tmax and tmin else np.nan,
            f'wind_avg_{lag}d': np.mean(wind) if wind else np.nan,
        }
    except Exception:
        cols = [f'precip_sum_{lag}d', f'precip_max_{lag}d', f'temp_max_{lag}d',
                f'temp_min_{lag}d', f'temp_range_{lag}d', f'wind_avg_{lag}d']
        return {c: np.nan for c in cols}

weather_keys = all_data[[STATION_COL, LAT_COL, LON_COL, DATE_COL]].drop_duplicates()
print(f'Unique (station, date) combos: {len(weather_keys)}')

In [ ]:
CHUNK = 50
ckpt = f'{CKPT_DIR}/weather.parquet'

if os.path.exists(ckpt):
    weather_df = pd.read_parquet(ckpt)
    print(f'[LOADED] weather: {len(weather_df)} rows')
else:
    parts = []
    total = len(weather_keys)
    for i in range(0, total, CHUNK):
        chunk = weather_keys.iloc[i:i+CHUNK].copy()
        for lag in [7, 14, 30]:
            wx = chunk.apply(lambda r: fetch_weather(
                r[LAT_COL], r[LON_COL], str(r[DATE_COL]), lag), axis=1)
            for col_name in wx.iloc[0].keys():
                chunk[col_name] = wx.apply(lambda x: x.get(col_name, np.nan))
        parts.append(chunk)
        pct = min(100, (i + CHUNK) / total * 100)
        print(f'  {pct:5.1f}%  ({min(i+CHUNK, total)}/{total})', end='\r')

        if len(parts) % 10 == 0:
            pd.concat(parts).to_parquet(f'{CKPT_DIR}/weather_partial.parquet', index=False)

    weather_df = pd.concat(parts, ignore_index=True)
    weather_df.to_parquet(ckpt, index=False)
    print(f'\n[SAVED] weather: {len(weather_df)} rows')

display(weather_df.describe())

### A4. OSM Pollution Proximity

In [ ]:
def fetch_osm(lat, lon, radius=5000):
    q = f"""[out:json][timeout:30];
    (node(around:{radius},{lat},{lon})["man_made"="mine"];
     way(around:{radius},{lat},{lon})["man_made"="wastewater_plant"];
     way(around:{radius},{lat},{lon})["landuse"="farmland"];
     way(around:{radius},{lat},{lon})["highway"];);
    out count;"""
    try:
        r = requests.get('http://overpass-api.de/api/interpreter',
                        params={'data': q}, timeout=60)
        r.raise_for_status()
        return int(r.json().get('elements', [{}])[0].get('tags', {}).get('total', 0))
    except Exception:
        return 0

ckpt = f'{CKPT_DIR}/osm.parquet'
if os.path.exists(ckpt):
    osm_df = pd.read_parquet(ckpt)
    print(f'[LOADED] osm: {len(osm_df)} rows')
else:
    print(f'Extracting OSM for {len(unique_stations)} stations (3 radii)...')
    osm_df = unique_stations.copy().reset_index(drop=True)
    for radius in [1000, 5000, 10000]:
        osm_df[f'osm_total_{radius}m'] = osm_df.apply(
            lambda r: fetch_osm(r[LAT_COL], r[LON_COL], radius), axis=1)
        print(f'  radius {radius}m done')
    osm_df.to_parquet(ckpt, index=False)
    print(f'[SAVED] {len(osm_df)} rows')

display(osm_df.describe())

---
## Part B: Heavy Downloads (wget -> extract -> rm)

Fill download URLs when ready. Pattern:
```python
!wget -q URL -O /kaggle/working/big_file.zip
# ... extract per station ...
!rm /kaggle/working/big_file.zip
```

In [ ]:
heavy_names = ['hydroatlas', 'riveratlas', 'sanlc', 'worldpop']
heavy_dfs = {}

for ds in heavy_names:
    ckpt = f'{CKPT_DIR}/{ds}.parquet'
    if os.path.exists(ckpt):
        heavy_dfs[ds] = pd.read_parquet(ckpt)
        print(f'[LOADED] {ds}: {len(heavy_dfs[ds])} rows')
    else:
        print(f'[TODO]   {ds}: not extracted yet')
        heavy_dfs[ds] = unique_stations.copy()
        heavy_dfs[ds][f'{ds}_placeholder'] = np.nan

---
## Merge All External Features

In [ ]:
station_keys = [STATION_COL, LAT_COL, LON_COL]
static_merged = unique_stations.copy()

source_dfs = [
    (elev_df, 'Elevation'), (soil_df, 'SoilGrids'), (osm_df, 'OSM'),
] + [(heavy_dfs[d], d.title()) for d in heavy_names]

feature_counts = {}  # for the bar chart

for src_df, name in source_dfs:
    new_cols = [c for c in src_df.columns if c not in station_keys]
    if new_cols:
        static_merged = static_merged.merge(
            src_df[station_keys + new_cols], on=station_keys, how='left')
        feature_counts[name] = len(new_cols)
        print(f'  + {name}: {len(new_cols)} features')

print(f'\nStatic per-station features: {static_merged.shape}')

# Merge static onto train / val
train_enriched = train_base.merge(static_merged, on=station_keys, how='left')
val_enriched   = val_base.merge(static_merged, on=station_keys, how='left')

# Merge temporal weather (per sample)
wx_merge_keys = [STATION_COL, LAT_COL, LON_COL, DATE_COL]
wx_cols = [c for c in weather_df.columns if c not in wx_merge_keys]
feature_counts['Weather'] = len(wx_cols)

train_enriched = train_enriched.merge(weather_df[wx_merge_keys + wx_cols],
                                       on=wx_merge_keys, how='left')
val_enriched   = val_enriched.merge(weather_df[wx_merge_keys + wx_cols],
                                     on=wx_merge_keys, how='left')

print(f'\nTrain enriched: {train_enriched.shape}')
print(f'Val enriched:   {val_enriched.shape}')
print(f'New features:   {train_enriched.shape[1] - train_base.shape[1]}')

---
## FIGURE 1: Features Added per Data Source

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

names  = list(feature_counts.keys())
counts = list(feature_counts.values())
palette = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0',
           '#F44336', '#00BCD4', '#795548', '#607D8B']

bars = ax.bar(names, counts, color=palette[:len(names)],
              edgecolor='white', linewidth=1.5)
ax.bar_label(bars, fontsize=12, fontweight='bold')

ax.set_ylabel('Number of Features')
ax.set_title(f'Features Added by Each Data Source  (total = {sum(counts)})')
plt.xticks(rotation=25, ha='right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_01_features_per_source.png', dpi=150, bbox_inches='tight')
plt.show()

---
## FIGURE 2: Data Coverage Map (green = has data, red = missing)

In [ ]:
check_cols = {
    'Elevation': 'elevation_m',
    'SoilGrids': 'soil_phh2o',
    'OSM':       'osm_total_5000m',
}

n_panels = len(check_cols)
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))
if n_panels == 1:
    axes = [axes]

for i, (src, col) in enumerate(check_cols.items()):
    ax = axes[i]
    if col in static_merged.columns:
        has = static_merged[col].notna()
        clr = ['#4CAF50' if v else '#F44336' for v in has]
        ax.scatter(static_merged[LON_COL], static_merged[LAT_COL],
                  c=clr, s=50, alpha=0.85, edgecolors='gray', linewidths=0.3)
        n_ok = has.sum()
        ax.set_title(f'{src}\n{n_ok}/{len(has)} stations', fontsize=12)
    else:
        ax.text(0.5, 0.5, 'Not extracted', transform=ax.transAxes,
                ha='center', va='center', fontsize=14, color='gray')
        ax.set_title(src, fontsize=12)
    ax.set_xlim(16, 33);  ax.set_ylim(-35, -22)
    ax.set_xlabel('Longitude');  ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.2)

legend_el = [mpatches.Patch(facecolor='#4CAF50', label='Has data'),
             mpatches.Patch(facecolor='#F44336', label='Missing')]
fig.legend(handles=legend_el, loc='lower center', ncol=2, fontsize=11,
           bbox_to_anchor=(0.5, -0.05))

fig.suptitle('Data Coverage per Source', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_01_data_coverage_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Extraction Summary

In [ ]:
rows = []
for src, col in check_cols.items():
    if col in static_merged.columns:
        ok = int(static_merged[col].notna().sum())
        tot = len(static_merged)
        rows.append({'Source': src, 'Features': feature_counts.get(src, 0),
                     'Coverage': f'{ok}/{tot}', 'Pct': f'{ok/tot*100:.0f}%',
                     'Status': 'OK' if ok == tot else 'Partial'})

rows.append({'Source': 'Weather', 'Features': feature_counts.get('Weather', 0),
             'Coverage': f'{len(weather_df)}/{len(weather_keys)}',
             'Pct': f'{len(weather_df)/max(1,len(weather_keys))*100:.0f}%',
             'Status': 'OK'})

for ds in heavy_names:
    rows.append({'Source': ds.title(), 'Features': feature_counts.get(ds.title(), 0),
                'Coverage': '—', 'Pct': '—', 'Status': 'TODO'})

display(pd.DataFrame(rows))

---
## Save

In [ ]:
train_enriched.to_parquet(f'{WORK_DIR}/train_enriched.parquet', index=False)
val_enriched.to_parquet(f'{WORK_DIR}/val_enriched.parquet', index=False)

print(f'[OK] train_enriched.parquet: {train_enriched.shape}')
print(f'[OK] val_enriched.parquet:   {val_enriched.shape}')
print(f'\nNew features added: {train_enriched.shape[1] - train_base.shape[1]}')